In [48]:
import os
import logging
import numpy as np
import nest_asyncio

In [2]:
from dotenv import load_dotenv
from openai import AzureOpenAI
from pathlib import Path

In [17]:
from lightrag import LightRAG, QueryParam
from lightrag.utils import EmbeddingFunc
from lightrag.kg.shared_storage import initialize_pipeline_status

In [3]:
logging.basicConfig(level=logging.INFO)

In [4]:
load_dotenv()

True

In [37]:
AZURE_OPENAI_API_VERSION = os.getenv("AZURE_OPENAI_API_VERSION")
AZURE_OPENAI_DEPLOYMENT = os.getenv("AZURE_OPENAI_DEPLOYMENT")
AZURE_OPENAI_API_KEY = os.getenv("AZURE_OPENAI_API_KEY")
AZURE_OPENAI_ENDPOINT = os.getenv("AZURE_OPENAI_ENDPOINT")

AZURE_EMBEDDING_DEPLOYMENT = os.getenv("AZURE_EMBEDDING_DEPLOYMENT")
AZURE_EMBEDDING_API_VERSION = os.getenv("AZURE_EMBEDDING_API_VERSION")

DATABASE_FILES_DIR = Path("database_files")
WORKING_DIR = Path("working_dir")

EMBEDDING_DIMENSION = 1536

In [6]:
azure_openai_client = AzureOpenAI(api_version=AZURE_OPENAI_API_VERSION, azure_endpoint=AZURE_OPENAI_ENDPOINT, api_key=AZURE_OPENAI_API_KEY)

In [7]:
sql_files = []
if DATABASE_FILES_DIR.exists() and DATABASE_FILES_DIR.is_dir():
    for sql_file in DATABASE_FILES_DIR.rglob("*.sql"):
        with open(sql_file, encoding="utf-8") as f:
            content = f.read()

        sql_files.append((sql_file, content))

In [8]:
system_prompt = """
You are an expert database documentation specialist. Your role is to create comprehensive, detailed documentation for database objects based on DDL (Data Definition Language) statements provided by users.

## Core Instructions

When a user provides DDL for any database object, create thorough documentation that extracts and utilizes ALL available information including comments, constraints, metadata, and specifications. Provide the most complete documentation possible - no detail is too small.

## Documentation Structure

Generate documentation with these sections:

### Object Overview
- Identify the type of database object (table, view, procedure, function, index, trigger, etc.)
- Explain the primary purpose and role within the database schema
- Provide business context and main use cases

### Detailed Structure & Components
- **For tables/views:** Document every column with complete details from DDL comments and constraints
- **For procedures/functions:** Detail all parameters, return types, and logic flow
- **For indexes:** Specify all columns covered, index type, purpose, and performance impact
- **For other objects:** Cover all relevant structural elements with full specifications

### Component Analysis (Leverage ALL DDL Comments)
- Extract business meaning and purpose from inline comments
- Document complete data type specifications (precision, scale, length)
- List all validation rules, constraints, and business logic
- Identify required vs optional elements with reasoning
- Explain default values, their significance, and business rationale
- Note any special handling, edge cases, or implementation details

### Complete Relationship Mapping
- Map all foreign key relationships with detailed explanations
- Identify self-referencing relationships and hierarchical structures
- Document dependencies on other database objects
- List objects that depend on this one
- Provide impact analysis for changes or cascading operations

### Comprehensive Constraints & Rules
- Document every constraint with business justification
- List all business rules enforced at database level
- Note security, access, and data integrity considerations
- Explain performance implications and optimization details

### Usage Patterns & Integration
- Describe how the object fits into larger business processes
- Detail common and advanced interaction patterns
- Identify query patterns this object supports
- Note performance characteristics and tuning considerations
- Explain integration points with applications

### Implementation Details
- Document storage specifications and logging settings
- Note any special database features utilized
- Include maintenance and operational considerations

## Quality Standards

- **Comprehensiveness:** Extract every piece of information from the DDL
- **Clarity:** Use clear, professional language accessible to technical and business users
- **Structure:** Organize with clear markdown headings for easy scanning
- **Accuracy:** Base all documentation strictly on the provided DDL
- **Detail:** Include all constraints, comments, data types, and specifications

## Output Format

Provide well-structured markdown documentation with clear headings. Make the documentation comprehensive yet scannable, suitable for database developers, business analysts, application developers, data architects, and database administrators.

Always begin your response with a clear heading identifying the database object name and type.
"""

In [ ]:
for md_file in DATABASE_FILES_DIR.rglob("*.md"):
    try:
        md_file.unlink(missing_ok=True)
    except PermissionError:
        logging.error(f"Permission denied deleting file {md_file}")
    except Exception as e:
        logging.error(f"Unexpected error deleting file {md_file}: {e}")

In [10]:
for sql_file, content in sql_files:
    user_prompt = content.strip()
    messages = []
    messages.append({"role": "system", "content": system_prompt})
    messages.append({"role": "user", "content": user_prompt})
    chat_completion = azure_openai_client.chat.completions.create(
            model=AZURE_OPENAI_DEPLOYMENT,
            messages=messages,
            temperature=0,
            top_p=1,
            n=1,
        ).choices[0].message.content
    with open(sql_file.with_suffix(".md"), "w") as f:
        f.write(chat_completion)


INFO:httpx:HTTP Request: POST https://heihachi-test-1-resource.openai.azure.com/openai/deployments/gpt-4.1-mini/chat/completions?api-version=2025-01-01-preview "HTTP/1.1 200 OK"
INFO:httpx:HTTP Request: POST https://heihachi-test-1-resource.openai.azure.com/openai/deployments/gpt-4.1-mini/chat/completions?api-version=2025-01-01-preview "HTTP/1.1 200 OK"
INFO:httpx:HTTP Request: POST https://heihachi-test-1-resource.openai.azure.com/openai/deployments/gpt-4.1-mini/chat/completions?api-version=2025-01-01-preview "HTTP/1.1 200 OK"
INFO:httpx:HTTP Request: POST https://heihachi-test-1-resource.openai.azure.com/openai/deployments/gpt-4.1-mini/chat/completions?api-version=2025-01-01-preview "HTTP/1.1 200 OK"
INFO:httpx:HTTP Request: POST https://heihachi-test-1-resource.openai.azure.com/openai/deployments/gpt-4.1-mini/chat/completions?api-version=2025-01-01-preview "HTTP/1.1 200 OK"
INFO:httpx:HTTP Request: POST https://heihachi-test-1-resource.openai.azure.com/openai/deployments/gpt-4.1-min

In [ ]:
for md_file in DATABASE_FILES_DIR.rglob("*.md"):
    try:
        target_file = Path(WORKING_DIR) / md_file.name
        target_file.write_text(md_file.read_text(encoding="utf-8"), encoding="utf-8")
    except PermissionError:
        logging.error(f"Permission denied copying file {md_file} to {target_file}")
    except Exception as e:
        logging.error(f"Unexpected error copying file {md_file} to {target_file}: {e}")

In [11]:
async def llm_model_func(
    prompt, system_prompt=None, history_messages=[], keyword_extraction=False, **kwargs
) -> str:
    client = AzureOpenAI(
        api_key=AZURE_OPENAI_API_KEY,
        api_version=AZURE_OPENAI_API_VERSION,
        azure_endpoint=AZURE_OPENAI_ENDPOINT,
    )

    messages = []
    if system_prompt:
        messages.append({"role": "system", "content": system_prompt})
    if history_messages:
        messages.extend(history_messages)
    messages.append({"role": "user", "content": prompt})

    chat_completion = client.chat.completions.create(
        model=AZURE_OPENAI_DEPLOYMENT,
        messages=messages,
        temperature=kwargs.get("temperature", 0),
        top_p=kwargs.get("top_p", 1),
        n=kwargs.get("n", 1),
    )
    return chat_completion.choices[0].message.content

In [14]:
async def embedding_func(texts: list[str]) -> np.ndarray:
    client = AzureOpenAI(
        api_key=AZURE_OPENAI_API_KEY,
        api_version=AZURE_EMBEDDING_API_VERSION,
        azure_endpoint=AZURE_OPENAI_ENDPOINT,
    )
    embedding = client.embeddings.create(model=AZURE_EMBEDDING_DEPLOYMENT, input=texts)

    embeddings = [item.embedding for item in embedding.data]
    return np.array(embeddings)

In [40]:
async def initialize_rag():
    rag = LightRAG(
        working_dir=WORKING_DIR.name,
        llm_model_func=llm_model_func,
        embedding_func=EmbeddingFunc(
            embedding_dim=EMBEDDING_DIMENSION,
            max_token_size=8192,
            func=embedding_func,
        ),
    )

    await rag.initialize_storages()
    await initialize_pipeline_status()

    return rag

In [49]:
nest_asyncio.apply()

rag = await initialize_rag()

for md_file in WORKING_DIR.rglob("*.md"):
    print(md_file.relative_to(WORKING_DIR))
    with open(md_file, encoding="utf-8") as doc:
        rag.insert([doc.read()])

INFO:nano-vectordb:Init {'embedding_dim': 1536, 'metric': 'cosine', 'storage_file': 'working_dir/vdb_entities.json'} 0 data
INFO:nano-vectordb:Init {'embedding_dim': 1536, 'metric': 'cosine', 'storage_file': 'working_dir/vdb_relationships.json'} 0 data
INFO:nano-vectordb:Init {'embedding_dim': 1536, 'metric': 'cosine', 'storage_file': 'working_dir/vdb_chunks.json'} 0 data
Rerank is enabled but no rerank_model_func provided. Reranking will be skipped.


hr.jhist_department_ix.md
hr.employees.md
hr.locations.md
hr.loc_city_ix.md
hr.regions.md
hr.jhist_job_ix.md
hr.job_history.md
hr.loc_state_province_ix.md
hr.emp_details_view.md
hr.emp_job_is.md


/home/tothi/python/dbchat3/.venv/lib/python3.11/site-packages/networkx/readwrite/graphml.py:652: RuntimeWarning: coroutine 'LightRAG.ainsert' was never awaited
  str(k), self.attr_type(k, scope, v), str(v), scope, default


hr.departments.md
hr.dept_location_ix.md
hr.emp_manager_ix.md
hr.emp_name_ix.md
hr.emp_department_ix.md
hr.loc_country_ix.md
hr.countries.md
hr.jobs.md
hr.jhist_employee_ix.md


In [51]:
query_text = "What tables the emp_details_view uses?"

In [52]:
print("Result (Naive):")
print(rag.query(query_text, param=QueryParam(mode="naive")))

Result (Naive):


Rerank is enabled but no rerank model is configured. Please set up a rerank model or set enable_rerank=False in query parameters.


# Tables Used by `hr.EMP_DETAILS_VIEW`

The `hr.EMP_DETAILS_VIEW` is a read-only view designed to provide a comprehensive snapshot of employee information by consolidating data from multiple related tables within the HR schema. According to the documentation, this view depends on the following six base tables:

1. **`hr.employees`**  
   This table provides core employee details such as employee ID, job ID, manager ID, department ID, first and last names, salary, and commission percentage.

2. **`hr.departments`**  
   Supplies department-related information including department ID, department name, and location ID.

3. **`hr.jobs`**  
   Contains job role details, including job ID and job title.

4. **`hr.locations`**  
   Provides location data such as location ID, city, and state or province.

5. **`hr.countries`**  
   Offers country-level information, including country ID and country name.

6. **`hr.regions`**  
   Contains regional data, including region name associated with coun

In [53]:
print("\nResult (Local):")
print(rag.query(query_text, param=QueryParam(mode="local")))


Result (Local):


Rerank is enabled but no rerank model is configured. Please set up a rerank model or set enable_rerank=False in query parameters.


# Tables Used by hr.EMP_DETAILS_VIEW

The `hr.EMP_DETAILS_VIEW` is a consolidated, read-only database view within the HR schema that integrates detailed employee information by joining multiple related tables. Specifically, it depends on the following six base tables:

1. **employees**  
   This table provides core employee data such as employee ID, job ID, manager ID, department ID, first and last names, salary, and commission percentage.

2. **departments**  
   This table supplies department-related information including department ID, department name, manager assignments, and location IDs.

3. **jobs**  
   The jobs table contributes job role details, including job titles associated with employees.

4. **locations**  
   This table contains data about physical locations, such as city, state or province, and country IDs, which are linked to departments.

5. **countries**  
   The countries table provides country-level information, including country names, linked via location data.



In [54]:
print("\nResult (Global):")
print(rag.query(query_text, param=QueryParam(mode="global")))


Result (Global):


Rerank is enabled but no rerank model is configured. Please set up a rerank model or set enable_rerank=False in query parameters.


## Tables Used by hr.EMP_DETAILS_VIEW

The `hr.EMP_DETAILS_VIEW` is a read-only consolidated database view within the HR schema that provides a comprehensive snapshot of employee data. It achieves this by joining multiple base tables to integrate detailed employee information along with related organizational and geographic context.

Specifically, the view depends on the following six base tables within the `hr` schema:

- **Employees Table (`hr.employees`)**: Provides core employee data such as employee ID, job ID, manager ID, department ID, first and last names, salary, and commission percentage.
- **Departments Table (`hr.departments`)**: Supplies department-related information including department ID, department name, and location ID.
- **Jobs Table (`hr.jobs`)**: Contains job titles and job-related information linked to employees via job IDs.
- **Locations Table (`hr.locations`)**: Stores geographic location data such as city, state or province, and country ID, referenced by depar

In [55]:
print("\nResult (Hybrid):")
print(rag.query(query_text, param=QueryParam(mode="hybrid")))


Result (Hybrid):


Rerank is enabled but no rerank model is configured. Please set up a rerank model or set enable_rerank=False in query parameters.


# Tables Used by hr.EMP_DETAILS_VIEW

The `hr.EMP_DETAILS_VIEW` is a consolidated, read-only database view within the HR schema that integrates detailed employee information by joining multiple related tables. Specifically, it depends on the following six base tables:

1. **employees**  
   This table provides core employee data such as employee identifiers, job roles, manager relationships, department assignments, personal details (first and last names), salary, and commission percentages.

2. **departments**  
   The departments table supplies department-related information including department identifiers, names, managerial assignments, and location identifiers.

3. **jobs**  
   This table contains job-related information such as job IDs and job titles, linking employees to their specific job roles.

4. **locations**  
   Locations provide geographic data about physical locations where departments are situated, including city and state or province details.

5. **countries**  
   Th

In [58]:
print("\nResult (Hybrid):")
print(rag.query("list all the objects that use the departments table, just list them, do not add any other information", param=QueryParam(mode="hybrid")))


Result (Hybrid):


Rerank is enabled but no rerank model is configured. Please set up a rerank model or set enable_rerank=False in query parameters.


- HR.JOB_HISTORY  
- HR.EMPLOYEES  
- HR.EMP_DETAILS_VIEW  
- LOCATIONS Table  
- Queries, Views, and Stored Procedures  
- Organizational Hierarchy  
- Business Processes  
- Integration Points  
- Reporting Tools  
- Applications Querying Employee Data


In [59]:
print("\nResult (Hybrid):")
print(rag.query("how does the LOCATIONS table uses the departments table?", param=QueryParam(mode="hybrid")))


Result (Hybrid):


Rerank is enabled but no rerank model is configured. Please set up a rerank model or set enable_rerank=False in query parameters.


# Relationship Between LOCATIONS and DEPARTMENTS Tables

The LOCATIONS table primarily stores detailed information about physical locations such as offices, warehouses, and production sites within the company. It includes columns like LOCATION_ID (primary key), CITY, STATE_PROVINCE, COUNTRY_ID (foreign key to COUNTRIES table), and others that describe the geographic and address details of each location.

The DEPARTMENTS table contains information about various organizational departments, including their unique identifiers, names, assigned managers, and importantly, a LOCATION_ID column. This LOCATION_ID in the DEPARTMENTS table is a foreign key referencing the LOCATION_ID in the LOCATIONS table. This relationship associates each department with a specific physical location where it is situated.

# How LOCATIONS Uses DEPARTMENTS

While the LOCATIONS table itself does not directly use the DEPARTMENTS table, it is referenced by the DEPARTMENTS table to provide location details for departm

In [61]:
print("\nResult (Hybrid):")
print(rag.query("write a query that list all the employees by locations and countries. required field: location id, county id, employee id", param=QueryParam(mode="hybrid")))


Result (Hybrid):


Rerank is enabled but no rerank model is configured. Please set up a rerank model or set enable_rerank=False in query parameters.


To list all employees by their locations and countries with the required fields (LOCATION_ID, COUNTRY_ID, EMPLOYEE_ID), you can join the HR.EMPLOYEES table with the HR.DEPARTMENTS table (to get LOCATION_ID) and then join with the HR.LOCATIONS table (to get COUNTRY_ID). 

Here is a sample SQL query:

```sql
SELECT 
    d.LOCATION_ID,
    l.COUNTRY_ID,
    e.EMPLOYEE_ID
FROM 
    HR.EMPLOYEES e
JOIN 
    HR.DEPARTMENTS d ON e.DEPARTMENT_ID = d.DEPARTMENT_ID
JOIN 
    HR.LOCATIONS l ON d.LOCATION_ID = l.LOCATION_ID
ORDER BY 
    d.LOCATION_ID, l.COUNTRY_ID, e.EMPLOYEE_ID;
```

### Explanation:
- `HR.EMPLOYEES` contains employee data including `EMPLOYEE_ID` and `DEPARTMENT_ID`.
- `HR.DEPARTMENTS` contains `DEPARTMENT_ID` and `LOCATION_ID`.
- `HR.LOCATIONS` contains `LOCATION_ID` and `COUNTRY_ID`.
- The joins link employees to their departments, departments to their locations, and locations to their countries.
- The query selects the required fields and orders the results by location, count